# BehaveGuard — Behavioral Authentication Demo

End-to-end reproducible demo of the **BehaveGuard** behavioral-authentication pipeline:

1. Load the `Behaveguard-client.xlsx` workbook (Kaggle dataset input).
2. Overview of the dataset and its quality caveats.
3. Canonical feature engineering keyboard + mouse behavior.
4. Time-disjoint window pseudo-sessions for development evaluation.
5. Classical identification/verification models (logistic, kNN, RF, Extra-Trees, RBF-SVM grid).
6. Modality ablation (keyboard-only / mouse-only / multimodal).
7. Inter-profile cosine similarity matrix.
8. Experimental BiLSTM + TCN fusion neural model.
9. Personal leave-one-session-out neural verifier for `saruman`.

> **Validity note:** the workbook contains one real session per identity (10 sessions / 9 unique people after merging the `elrond`/`akshit` aliases into `saruman`). All windows originate from a single parent session, so reported accuracy measures **development separability**, not cross-day production FAR/FRR. Do not promote these numbers to an operational claim.

In [ ]:
# BehaveGuard ships no pip package; this notebook is self-contained.
# Kaggle already provides: numpy, pandas, scikit-learn, torch, matplotlib, seaborn.
import json, math, copy, hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.base import clone

try:
    import torch
    from torch import nn
    print('torch', torch.__version__)
except Exception as e:
    print('torch unavailable:', e)

sns.set_theme(style='whitegrid')
RNG = 42
np.random.seed(RNG)

PROFILE_ALIASES = {'elrond': 'saruman', 'akshit': 'saruman'}

## 1. Locate the dataset

Two supported layouts:

- **Cleaned CSVs (preferred)** — produced by `scripts/export_kaggle_dataset.py`,
  dropped `IKI_Sequences`/`Trigraphs` and leakage-prone columns. On Kaggle, attach
  a dataset named `behaveguard-client` whose contents are these CSVs.
- **Raw `Behaveguard-client.xlsx`** — used as a fallback when CSVs are not present.

Both paths build the same `workbook` dict so the rest of the notebook is
format-agnostic.

In [ ]:
def load_workbook():
    csv_dirs = [
        Path('/kaggle/input/behaveguard-client'),
        Path('../input/behaveguard-client'),
        Path('kaggle_dataset'),
        Path('../kaggle_dataset'),
        Path('../input/behaveguard-client/kaggle_dataset'),
    ]
    csv_dir = next((d for d in csv_dirs if (d / 'Sessions.csv').exists()), None)
    if csv_dir is not None:
        print('Loading cleaned CSVs from:', csv_dir)
        out = {}
        for sheet in [
            'Sessions', 'KeyEvents', 'MousePassive', 'TrackSamples',
            'TrackTrials', 'DotTrials', 'DragTrials',
        ]:
            out[sheet] = pd.read_csv(csv_dir / f'{sheet}.csv')
        # normalize boolean column read back as 0/1
        out['DragTrials']['success'] = out['DragTrials']['success'].astype(bool)
        out['KeyEvents']['shift_held'] = out['KeyEvents']['shift_held'].astype(bool)
        return out
    candidates = [
        Path('/kaggle/input/behaveguard-client/Behaveguard-client.xlsx'),
        Path('../input/behaveguard-client/Behaveguard-client.xlsx'),
        Path('Behaveguard-client.xlsx'),
        Path('../Behaveguard-client.xlsx'),
    ]
    xlsx_path = next((p for p in candidates if p.exists()), None)
    if xlsx_path is None:
        raise FileNotFoundError(
            'Attach a behaveguard-client Kaggle Dataset (cleaned CSVs or '
            'Behaveguard-client.xlsx), or run scripts/export_kaggle_dataset.py.')
    print('Loading raw workbook:', xlsx_path)
    return pd.read_excel(xlsx_path, sheet_name=None)

workbook = load_workbook()
for name, df in workbook.items():
    print(f'{name:14s} {df.shape}')

## 2. Dataset overview

The workbook has 9 sheets. Each row in `Sessions` is one labeled collection; the other sheets are per-event streams joined by `(subject_id, collected_at)`. `Siya` provided mouse but no key events.

In [ ]:
sessions = workbook['Sessions'].copy()
sessions['canonical'] = sessions.subject_id.map(lambda s: PROFILE_ALIASES.get(s.casefold(), s))
n_subjects = sessions.subject_id.nunique()
n_identities = sessions.canonical.nunique()
print(f'Subject labels : {n_subjects}')
print(f'Identities    : {n_identities}  (elrond + akshit merged into saruman)')
print(f'Sessions      : {len(sessions)}')
print()
total_keys = len(workbook['KeyEvents'])
missing_release = int(workbook['KeyEvents'].release_ts.isna().sum())
print(f'KeyEvents rows     : {total_keys:,}  (missing release_ts: {missing_release})')
print(f'MousePassive rows  : {len(workbook["MousePassive"]):,}')
print(f'TrackSamples rows  : {len(workbook["TrackSamples"]):,}')
print(f'DotTrials          : {len(workbook["DotTrials"])}')
print(f'DragTrials         : {len(workbook["DragTrials"])}')
print()
print('Redundant sheets dropped from the dataset: IKI_Sequences, Trigraphs '
      '(derivable from KeyEvents).')
sessions[['subject_id','canonical','collected_at','duration_ms','backspace_count']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ke = workbook['KeyEvents'].dropna(subset=['press_ts','release_ts']).copy()
ke['dwell'] = ke.release_ts - ke.press_ts
axes[0].hist(ke.dwell.clip(0, 400), bins=40, color='#4C72B0')
axes[0].set_title('Key dwell (ms)'); axes[0].set_xlabel('dwell_ms')
mp = workbook['MousePassive'].sort_values(['subject_id','collected_at','ts'])
dt = mp.groupby(['subject_id','collected_at']).ts.diff().fillna(1)
dist = np.hypot(mp.dx, mp.dy)
speeds = (dist / dt).clip(0, 5)
axes[1].hist(speeds, bins=40, color='#55A868')
axes[1].set_title('Passive mouse speed (px/ms)'); axes[1].set_xlabel('speed')
dt_err = workbook['TrackSamples']
err = np.hypot(dt_err.cursor_x - dt_err.target_x, dt_err.cursor_y - dt_err.target_y)
axes[2].hist(err.clip(0, 150), bins=40, color='#C44E52')
axes[2].set_title('Pursuit tracking error (px)'); axes[2].set_xlabel('error_px')
plt.tight_layout(); plt.show()

## 3. Canonical feature engineering

Replicate `src/behaveguard/features.py`: robust per-stream statistics (mean, std, p10/p50/p90, IQR) over keyboard dwell/IKI/flight, passive mouse kinematics, dot/drag trials, and per-pattern pursuit-tracking derived metrics. **No leakage features** — no absolute coordinates, wall-clock time, or subject label.

In [ ]:
def _finite(values):
    out = []
    for v in values:
        try:
            x = float(v)
            if math.isfinite(x):
                out.append(x)
        except (TypeError, ValueError):
            pass
    return np.asarray(out, dtype=np.float64)

def _stats(prefix, values):
    arr = _finite(values)
    if arr.size == 0:
        return {f'{prefix}_{n}': 0.0 for n in ('mean','std','p10','p50','p90','iqr')}
    q10,q25,q50,q75,q90 = np.percentile(arr,[10,25,50,75,90])
    return {f'{prefix}_mean': float(np.mean(arr)), f'{prefix}_std': float(np.std(arr)),
            f'{prefix}_p10': float(q10), f'{prefix}_p50': float(q50),
            f'{prefix}_p90': float(q90), f'{prefix}_iqr': float(q75-q25)}

def _path_features(points, prefix):
    if len(points) < 2:
        return {**_stats(f'{prefix}_speed', []), **_stats(f'{prefix}_turn', []),
                f'{prefix}_pause_ratio': 0.0}
    dx, dy, dt = [], [], []
    for a, b in zip(points, points[1:]):
        d_t = max(float(b.get('ts',0))-float(a.get('ts',0)), 1.0)
        # Prefer stored relative displacement (dx/dy) when available; absolute
        # x/y is excluded from the cleaned dataset to avoid device leakage.
        if 'dx' in b:
            dx.append(float(b.get('dx',0))); dy.append(float(b.get('dy',0)))
        else:
            dx.append(float(b.get('x', b.get('cursor_x',0))) - float(a.get('x', a.get('cursor_x',0))))
            dy.append(float(b.get('y', b.get('cursor_y',0))) - float(a.get('y', a.get('cursor_y',0))))
        dt.append(d_t)
    speeds = np.hypot(dx, dy) / (np.asarray(dt)/1000.0)
    angles = np.unwrap(np.arctan2(dy, dx))
    turns = np.abs(np.diff(angles)) if len(angles) > 1 else np.asarray([])
    return {**_stats(f'{prefix}_speed', speeds), **_stats(f'{prefix}_turn', turns),
            f'{prefix}_pause_ratio': float(np.mean(speeds < 30)) if speeds.size else 0.0}

def extract_features(session):
    keyboard = session.get('keyboard') or {}
    events = sorted(keyboard.get('events') or [], key=lambda e: float(e.get('press_ts',0)))
    dwell = [float(e['release_ts'])-float(e['press_ts']) for e in events if e.get('release_ts') is not None]
    iki = [float(b.get('press_ts',0))-float(a.get('press_ts',0)) for a,b in zip(events, events[1:])]
    flight = [float(b.get('press_ts',0))-float(a.get('release_ts', a.get('press_ts',0)))
              for a,b in zip(events, events[1:]) if a.get('release_ts') is not None]
    feats = {
        **_stats('key_dwell', dwell), **_stats('key_iki', iki), **_stats('key_flight', flight),
        'key_count': float(len(events)),
        'key_backspace_rate': float(sum(e.get('key_id')=='backspace' for e in events)/max(len(events),1)),
        'key_shift_rate': float(sum(bool(e.get('shift_held')) for e in events)/max(len(events),1)),
        'key_space_rate': float(sum(e.get('key_category')=='space' for e in events)/max(len(events),1)),
        'key_special_rate': float(sum(e.get('key_category')=='special' for e in events)/max(len(events),1)),
    }
    mouse = session.get('mouse') or {}
    passive = mouse.get('passive_points') or []
    feats.update(_path_features(passive, 'passive'))
    feats['passive_count'] = float(len(passive))
    dots = mouse.get('dot_trials') or []
    feats.update(_stats('dot_travel', (t.get('travel_time_ms') for t in dots)))
    feats.update(_stats('dot_error', (t.get('error_px') for t in dots)))
    feats.update(_stats('dot_submove',
        ((t.get('kinematics') or {}).get('sub_movement_count', t.get('sub_movement_count')) for t in dots)))
    feats.update(_stats('dot_hover',
        ((t.get('kinematics') or {}).get('hover_dwell_ms', t.get('hover_dwell_ms')) for t in dots)))
    drags = mouse.get('drag_trials') or []
    feats.update(_stats('drag_duration', (t.get('duration_ms') for t in drags)))
    feats.update(_stats('drag_submove',
        ((t.get('kinematics') or {}).get('sub_movement_count', t.get('sub_movement_count')) for t in drags)))
    feats['drag_success_rate'] = float(sum(bool(t.get('success')) for t in drags)/max(len(drags),1))
    tracks = mouse.get('track_trials') or []
    for pattern in ('sinusoidal','random_walk'):
        sel = [t for t in tracks if t.get('pattern')==pattern]
        for n in ('mean_error_px','rms_error_px','lag_ms','prediction_ratio','tremor_px',
                  'correlation_x','correlation_y','fatigue_delta_px'):
            feats.update(_stats(f'track_{pattern}_{n}',
                ((t.get('derived') or {}).get(n) for t in sel)))
    return {n: (float(v) if math.isfinite(float(v)) else 0.0) for n,v in sorted(feats.items())}

def feature_vector(feats, names):
    return np.asarray([float(feats.get(n, 0.0)) for n in names], dtype=np.float64)
print('Feature extraction helpers defined.')

## 4. Build canonical sessions from the workbook

Re-implement `src/behaveguard/importer.py` so the notebook needs no installed `behaveguard` package.

In [ ]:
def _records(frame):
    return json.loads(frame.where(pd.notna(frame), None).to_json(orient='records'))

def build_sessions(workbook):
    sessions_rows = workbook['Sessions']
    out = []
    for _, summary in sessions_rows.iterrows():
        source_label = str(summary['subject_id'])
        label = PROFILE_ALIASES.get(source_label.casefold(), source_label)
        collected = str(summary['collected_at'])
        def rows(sheet):
            frame = workbook[sheet]
            sel = frame[(frame.subject_id.astype(str)==source_label) &
                        (frame.collected_at.astype(str)==collected)]
            return _records(sel.drop(columns=['subject_id','collected_at'], errors='ignore'))
        key_events = rows('KeyEvents')
        for e in key_events:
            e.pop('dwell_ms', None)
        track_trials = rows('TrackTrials')
        track_samples = rows('TrackSamples')
        for trial in track_trials:
            ix = trial.get('trial_index')
            trial['samples'] = [s for s in track_samples if s.get('trial_index')==ix]
            trial['derived'] = {n: trial.pop(n) for n in list(trial)
                                if n in {'mean_error_px','rms_error_px','lag_ms','prediction_ratio',
                                         'tremor_px','correlation_x','correlation_y',
                                         'error_first_half_px','error_second_half_px','fatigue_delta_px'}}
        payload = {
            'subject_id': label, 'collected_at': collected,
            'duration_ms': float(summary['duration_ms']),
            'keyboard': {'events': key_events, 'pangram_text_length': 0,
                         'free_text_length': 0, 'extras': {}},
            'mouse': {'passive_points': rows('MousePassive'), 'dot_trials': rows('DotTrials'),
                      'drag_trials': rows('DragTrials'), 'track_trials': track_trials},
            'context': {'source': 'kaggle-demo'},
        }
        out.append({'label': label, 'collected_at': collected, 'payload': payload})
    return out

canon_sessions = build_sessions(workbook)
for row in canon_sessions:
    row['features'] = extract_features(row['payload'])
labels = [r['label'] for r in canon_sessions]
print('Built', len(canon_sessions), 'canonical sessions.')
print('Per-identity counts:', dict(Counter(labels)))
print('Engineered feature dimension:',
      len(sorted({n for r in canon_sessions for n in r['features']})))
canon_sessions[0]['features']

## 5. Time-disjoint window pseudo-sessions

Each real session is split chronologically into `WINDOW_COUNT` non-overlapping windows. Windows from one parent session never overlap, but they still share device/day, so all reported metrics are **development_only** evaluation.

In [ ]:
WINDOW_COUNT = 5

def _chunk(values, index, count):
    start = round(len(values) * index / count)
    end   = round(len(values) * (index + 1) / count)
    return values[start:end]

def window_session(session, index, count=WINDOW_COUNT):
    keyboard = session.get('keyboard') or {}
    mouse = session.get('mouse') or {}
    tracks = []
    for trial in mouse.get('track_trials') or []:
        item = copy.deepcopy(trial)
        item['samples'] = _chunk(item.get('samples') or [], index, count)
        item['derived'] = {}
        tracks.append(item)
    return {
        'collected_at': session.get('collected_at'),
        'duration_ms': float(session.get('duration_ms', 0)) / count,
        'keyboard': {'events': _chunk(keyboard.get('events') or [], index, count),
                     'pangram_text_length': 0, 'free_text_length': 0, 'extras': {}},
        'mouse': {'passive_points': _chunk(mouse.get('passive_points') or [], index, count),
                  'dot_trials': _chunk(mouse.get('dot_trials') or [], index, count),
                  'drag_trials': _chunk(mouse.get('drag_trials') or [], index, count),
                  'track_trials': tracks},
        'context': {'experimental_window': index, 'window_count': count},
    }

def build_window_dataset(canon_sessions, window_count=WINDOW_COUNT):
    windows = []
    for r in canon_sessions:
        for index in range(window_count):
            s = window_session(r['payload'], index, window_count)
            windows.append({'label': r['label'], 'fold': index,
                            'session': s, 'features': extract_features(s)})
    names = sorted({n for w in windows for n in w['features']})
    matrix = np.vstack([feature_vector(w['features'], names) for w in windows])
    labels = np.asarray([w['label'] for w in windows])
    folds = np.asarray([w['fold'] for w in windows])
    return windows, names, matrix, labels, folds

windows, FEATURE_NAMES, X, y_labels, FOLDS = build_window_dataset(canon_sessions)
print(f'Windows: {len(windows)} | features: {len(FEATURE_NAMES)} | identities: {len(set(y_labels))}')
print('Class counts:', dict(Counter(y_labels)))

## 6. Classical identification + verification evaluation

Five-fold window evaluation: each fold holds out one chronological window from every identity. Models compared: logistic regression, 3-NN, Random Forest, Extra Trees, and a representative **RBF-SVM** (further tuned below). Metrics: Top-1 / Top-3 identification, macro-F1, verification ROC-AUC (genuine vs. impostor decision scores), and equal-error rate (EER).

In [ ]:
def _decision_matrix(model, X, classes):
    d = np.asarray(model.decision_function(X))
    if len(classes) == 2 and d.ndim == 1:
        d = np.column_stack([-d, d])
    return d

def evaluate_model(estimator, X, y, folds):
    preds = np.empty(y.shape, dtype=object)
    classes = np.asarray(sorted(set(y)))
    score_rows = np.zeros((len(y), len(classes)))
    for fold in sorted(set(folds)):
        train = folds != fold
        test  = folds == fold
        model = clone(estimator).fit(X[train], y[train])
        preds[test] = model.predict(X[test])
        if hasattr(model, 'decision_function'):
            mc = model.classes_
            decision = _decision_matrix(model, X[test], mc)
        else:
            mc = model.classes_
            decision = model.predict_proba(X[test])
        for li, cid in enumerate(mc):
            score_rows[np.where(test)[0], np.where(classes==cid)[0][0]] = decision[:, li]
    targets = np.asarray([np.where(classes==c)[0][0] for c in y])
    top3 = np.argsort(score_rows, axis=1)[:, -3:]
    top3_acc = float(np.mean([t in c for t,c in zip(targets, top3)]))
    genuine, score = [], []
    for i, t in enumerate(targets):
        for j in range(len(classes)):
            genuine.append(int(j == t)); score.append(float(score_rows[i, j]))
    fpr, tpr, thr = roc_curve(genuine, score)
    fnr = 1 - tpr
    eer_i = int(np.nanargmin(np.abs(fpr - fnr)))
    return {
        'accuracy': round(float(accuracy_score(y, preds)), 4),
        'balanced_accuracy': round(float(balanced_accuracy_score(y, preds)), 4),
        'macro_f1': round(float(f1_score(y, preds, average='macro')), 4),
        'top3_accuracy': round(top3_acc, 4),
        'verification_auc': round(float(roc_auc_score(genuine, score)), 4),
        'eer': round(float((fpr[eer_i]+fnr[eer_i])/2), 4),
        'eer_threshold': round(float(thr[eer_i]), 4),
        'confusion_matrix': confusion_matrix(y, preds, labels=classes).tolist(),
        '_classes': classes.tolist(),
    }

candidates = {
    'logistic_regression': make_pipeline(RobustScaler(),
        LogisticRegression(C=1, max_iter=3000, class_weight='balanced')),
    'knn_3': make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=3, weights='distance')),
    'random_forest': RandomForestClassifier(n_estimators=350, min_samples_leaf=2,
        class_weight='balanced', random_state=RNG, n_jobs=-1),
    'extra_trees': ExtraTreesClassifier(n_estimators=350, min_samples_leaf=2,
        class_weight='balanced', random_state=RNG, n_jobs=-1),
    'svm_rbf': make_pipeline(RobustScaler(quantile_range=(10, 90)),
        SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced',
            decision_function_shape='ovr', probability=False)),
}
results = {name: evaluate_model(est, X, y_labels, FOLDS) for name, est in candidates.items()}
rows = []
for name, r in results.items():
    rows.append({'Model': name, 'Top-1': r['accuracy'], 'Top-3': r['top3_accuracy'],
                 'MacroF1': r['macro_f1'], 'VerifAUC': r['verification_auc'], 'EER': r['eer']})
pd.DataFrame(rows).sort_values('Top-1', ascending=False)

In [ ]:
best_name = max(results, key=lambda n: (results[n]['balanced_accuracy'], results[n]['verification_auc']))
print('Best classical model:', best_name)
classes = results[best_name]['_classes']
cm = np.asarray(results[best_name]['confusion_matrix'])
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title(f'Confusion matrix — {best_name}\n(window-disjoint 5-fold)')
plt.xlabel('predicted'); plt.ylabel('true'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 7. RBF-SVM grid tuning

Sweep `C in {0.1, 1, 10, 50}` × `gamma in {scale, 0.001, 0.01, 0.1}` with OvR decision function.

In [ ]:
svm_results = {}
for C in (0.1, 1.0, 10.0, 50.0):
    for gamma in ('scale', 0.001, 0.01, 0.1):
        name = f'svm_C{C}_g{gamma}'
        est = make_pipeline(RobustScaler(quantile_range=(10,90)),
            SVC(kernel='rbf', C=C, gamma=gamma, class_weight='balanced', decision_function_shape='ovr'))
        svm_results[name] = evaluate_model(est, X, y_labels, FOLDS)
        svm_results[name]['_classes'] = results[best_name]['_classes']
svm_rows = [{'Model': n, 'Top-1': r['accuracy'], 'Top-3': r['top3_accuracy'],
             'VerifAUC': r['verification_auc'], 'EER': r['eer']}
            for n, r in svm_results.items()]
pd.DataFrame(svm_rows).sort_values(['Top-1','VerifAUC'], ascending=False)

In [ ]:
best_svm_name = max(svm_results, key=lambda n: (svm_results[n]['balanced_accuracy'], svm_results[n]['verification_auc']))
print('Best RBF-SVM:', best_svm_name)
print(svm_results[best_svm_name])
merged = {**results, **svm_results}
overall_best = max(merged, key=lambda n: (merged[n]['balanced_accuracy'], merged[n]['verification_auc']))
print('\nOverall best:', overall_best)

## 8. Modality ablation

Compare keyboard-only, mouse-only, and full multimodal inputs using the tuned RBF-SVM. **Keyboard is currently more discriminative** (the `Siya` session has no key events), but fusion improves both identification and verification.

In [ ]:
best_svm_est = make_pipeline(RobustScaler(quantile_range=(10,90)),
    SVC(kernel='rbf', C=float(best_svm_name.split('_')[1][1:]),
       gamma=(0.01 if 'g0.01' in best_svm_name else 'scale'),
       class_weight='balanced', decision_function_shape='ovr'))
ablations = {}
for label, idx in {
    'keyboard_only':    [i for i,n in enumerate(FEATURE_NAMES) if n.startswith('key_')],
    'mouse_only':       [i for i,n in enumerate(FEATURE_NAMES) if not n.startswith('key_')],
    'full_multimodal':  list(range(len(FEATURE_NAMES))),
}.items():
    ablations[label] = evaluate_model(clone(best_svm_est), X[:, idx], y_labels, FOLDS)
ablation_rows = [{'Inputs': k, 'Top-1': v['accuracy'], 'Top-3': v['top3_accuracy'],
                  'VerifAUC': v['verification_auc'], 'EER': v['eer']}
                 for k, v in ablations.items()]
pd.DataFrame(ablation_rows)

## 9. Inter-profile cosine similarity

Robust-scaled full-session centroid per identity. Diagonal = 100%. Closest impostor pairs flag confusable identities in a deployed 1:N scorer.

In [ ]:
def profile_similarity(canon_sessions):
    names = sorted({n for r in canon_sessions for n in r['features']})
    M = np.vstack([feature_vector(r['features'], names) for r in canon_sessions])
    M = RobustScaler(quantile_range=(10,90)).fit_transform(M)
    labels = []
    centroids = []
    for label in sorted(set(r['label'] for r in canon_sessions)):
        idx = [i for i,r in enumerate(canon_sessions) if r['label']==label]
        labels.append(label); centroids.append(M[idx].mean(axis=0))
    C = np.vstack(centroids)
    norms = np.linalg.norm(C, axis=1, keepdims=True)
    N = np.divide(C, norms, out=np.zeros_like(C), where=norms!=0)
    sim = ((N @ N.T) + 1) * 50
    return labels, sim

sim_labels, sim_matrix = profile_similarity(canon_sessions)
plt.figure(figsize=(8,7))
sns.heatmap(sim_matrix, annot=True, fmt='.1f', cmap='magma',
            xticklabels=sim_labels, yticklabels=sim_labels)
plt.title('Inter-profile centroid similarity (% cosine)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
print('Closest impostor pairs:')
masked = sim_matrix - np.eye(len(sim_labels)) * 200
for i, l in enumerate(sim_labels):
    j = int(np.argmax(masked[i]))
    print(f'  {l:10s} <-> {sim_labels[j]:10s} {masked[i,j]:.2f}%')

## 10. BiLSTM + TCN fusion neural model

Replicate `src/behaveguard/neural.py`. Keyboard tower = 2-layer bi-LSTM over 6-channel key events, mouse tower = temporal-conv block over passive-mouse streams, fused with the engineered-feature projection into a 128-d L2-normalized embedding plus a classifier head.

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, input_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, hidden, 5, padding=2), nn.BatchNorm1d(hidden), nn.GELU(),
            nn.Conv1d(hidden, hidden, 3, padding=1), nn.BatchNorm1d(hidden), nn.GELU())
    def forward(self, x):
        return self.net(x.transpose(1,2)).mean(dim=-1)

class BehavioralSequenceNet(nn.Module):
    def __init__(self, feature_dim, n_classes, embedding_dim=128):
        super().__init__()
        self.keyboard = nn.LSTM(6, 48, num_layers=2, batch_first=True,
                                 bidirectional=True, dropout=0.15)
        self.mouse = TemporalBlock(6, 96)
        self.features = nn.Sequential(nn.Linear(feature_dim,128), nn.LayerNorm(128),
                                      nn.GELU(), nn.Dropout(0.15))
        self.fusion = nn.Sequential(nn.Linear(320,192), nn.GELU(), nn.Linear(192, embedding_dim))
        self.classifier = nn.Linear(embedding_dim, n_classes)
    def forward(self, k, m, f):
        ko, _ = self.keyboard(k)
        ke = ko.mean(dim=1)
        me = self.mouse(m)
        fe = self.features(f)
        emb = nn.functional.normalize(self.fusion(torch.cat([ke, me, fe], dim=-1)))
        return emb, self.classifier(emb)

def _key_code(v):
    digest = hashlib.blake2b(v.encode(), digest_size=2).digest()
    return int.from_bytes(digest, 'big') / 65535.0

def session_sequences(session, key_length=160, mouse_length=256):
    events = sorted(((session.get('keyboard') or {}).get('events') or []),
                    key=lambda e: e.get('press_ts',0))
    kb = np.zeros((key_length, 6), dtype=np.float32)
    for i, e in enumerate(events[:key_length]):
        prev = events[i-1] if i else e
        dwell = (e.get('release_ts') or e.get('press_ts',0)) - e.get('press_ts',0)
        iki = e.get('press_ts',0) - prev.get('press_ts',0)
        kb[i] = [_key_code(str(e.get('key_id',''))), dwell/500, iki/1000,
                 float(bool(e.get('shift_held'))), e.get('shift_hold_ms',0)/500, 1]
    points = (session.get('mouse') or {}).get('passive_points') or []
    ms = np.zeros((mouse_length, 6), dtype=np.float32)
    for i, p in enumerate(points[:mouse_length]):
        prev = points[i-1] if i else p
        dt = max(p.get('ts',0) - prev.get('ts',0), 1)
        dx, dy = p.get('dx',0), p.get('dy',0)
        ms[i] = [dx/100, dy/100, dt/100, np.hypot(dx,dy)/dt, p.get('pressure',0), 1]
    return kb, ms
print('Neural tower defined.')

In [ ]:
def train_neural_windows(windows, names, X, y, folds, epochs=25, seed=RNG):
    torch.manual_seed(seed); np.random.seed(seed)
    classes = np.asarray(sorted(set(y)))
    cidx = {v:i for i,v in enumerate(classes)}
    scaler = RobustScaler(quantile_range=(10,90)).fit(X[folds != folds.max()])
    scaled = scaler.transform(X).astype(np.float32)
    kb, ms = [], []
    for w in windows:
        k, m = session_sequences(w['session'], key_length=160, mouse_length=256)
        kb.append(k); ms.append(m)
    K = torch.tensor(np.asarray(kb), dtype=torch.float32)
    M = torch.tensor(np.asarray(ms), dtype=torch.float32)
    F = torch.tensor(scaled, dtype=torch.float32)
    T = torch.tensor([cidx[v] for v in y], dtype=torch.long)
    train_mask = torch.tensor(folds != folds.max())
    val_mask   = ~train_mask
    model = BehavioralSequenceNet(len(names), len(classes))
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    best_acc, best_state, history = -1.0, None, []
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        _, logits = model(K[train_mask], M[train_mask], F[train_mask])
        loss = nn.functional.cross_entropy(logits, T[train_mask], label_smoothing=0.05)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            _, vlogits = model(K[val_mask], M[val_mask], F[val_mask])
            acc = float((vlogits.argmax(1) == T[val_mask]).float().mean())
        history.append({'epoch': epoch+1, 'loss': round(float(loss.detach()),5),
                        'val_acc': round(acc,4)})
        if acc > best_acc:
            best_acc = acc
            best_state = {n: v.detach().cpu().clone() for n,v in model.state_dict().items()}
    return {'best_validation_accuracy': round(best_acc,4),
            'final_loss': history[-1]['loss'], 'history': history}

NEURAL_EPOCHS = 25
neural_result = train_neural_windows(windows, FEATURE_NAMES, X, y_labels, FOLDS, epochs=NEURAL_EPOCHS)
print('Neural development result:', neural_result)
h = pd.DataFrame(neural_result['history'])
fig, ax = plt.subplots(1,2, figsize=(11,3.5))
ax[0].plot(h.epoch, h.loss, marker='o'); ax[0].set_title('loss'); ax[0].set_xlabel('epoch')
ax[1].plot(h.epoch, h.val_acc, marker='o', color='#C44E52'); ax[1].set_title('held-window accuracy')
ax[1].set_xlabel('epoch'); ax[1].set_ylim(0,1.05)
plt.tight_layout(); plt.show()

## 11. Personal leave-one-session-out neural verifier for `saruman`

`elrond` + `akshit` were confirmed aliases of `saruman`, giving the only identity with multiple genuine sessions. Train a target-specific binary verifier that holds out one complete genuine parent session and a disjoint subset of impostor identities per fold (no held raw event/window enters that fold's training data). The personal score is **advisory** and never overrides the primary SVM/centroid decision.

In [ ]:
TARGET_LABEL = 'saruman'  # elrond + akshit are aliases of saruman; Akshat is a distinct person

def personal_folds(canon_sessions, target_label):
    genuine = [r for r in canon_sessions if r['label']==target_label]
    impostors = [r for r in canon_sessions if r['label']!=target_label]
    if len(genuine) < 3:
        raise ValueError('Need ≥3 genuine sessions for the personal verifier.')
    folds = []
    for i, held_g in enumerate(genuine):
        held_imp = impostors[i::len(genuine)]
        held = {r['collected_at'] for r in [held_g, *held_imp]}
        train = [r for r in canon_sessions if r['collected_at'] not in held]
        folds.append({'index': i+1, 'train': train, 'test_genuine': held_g,
                      'test_impostors': held_imp})
    return folds

def _win_examples(rows, target_label, window_count=4):
    out = []
    for r in rows:
        for wi in range(window_count):
            payload = window_session(r['payload'], wi, window_count)
            out.append({'target': int(r['label']==target_label), 'payload': payload,
                        'features': extract_features(payload)})
    return out

def _tensors(examples, names, scaler):
    kb, ms, vec, tgt = [], [], [], []
    for ex in examples:
        k, m = session_sequences(ex['payload'], key_length=160, mouse_length=256)
        kb.append(k); ms.append(m)
        vec.append(scaler.transform(feature_vector(ex['features'], names).reshape(1,-1))[0])
        tgt.append(ex['target'])
    return (torch.tensor(np.asarray(kb), dtype=torch.float32),
            torch.tensor(np.asarray(ms), dtype=torch.float32),
            torch.tensor(np.asarray(vec), dtype=torch.float32),
            torch.tensor(tgt, dtype=torch.long))

def _train_personal(examples, epochs=25, seed=RNG):
    torch.manual_seed(seed); np.random.seed(seed)
    names = sorted({n for ex in examples for n in ex['features']})
    M = np.vstack([feature_vector(ex['features'], names) for ex in examples])
    scaler = RobustScaler(quantile_range=(10,90)).fit(M)
    K, Ms, F, T = _tensors(examples, names, scaler)
    model = BehavioralSequenceNet(len(names), 2)
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=3e-3)
    weights = T.numel() / (2 * torch.bincount(T, minlength=2).float().clamp_min(1))
    losses = []
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        _, logits = model(K, Ms, F)
        loss = nn.functional.cross_entropy(logits, T, weight=weights, label_smoothing=0.04)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        opt.step()
        losses.append(round(float(loss.detach()),5))
    return model, scaler, names, losses

def _score(model, scaler, names, payload, window_count=4):
    examples = []
    for wi in range(window_count):
        w = window_session(payload, wi, window_count)
        examples.append({'payload': w, 'features': extract_features(w), 'target': 0})
    K, Ms, F, _ = _tensors(examples, names, scaler)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(K, Ms, F)[1], dim=1)[:,1].cpu().numpy()
    return float(np.mean(probs))

def _threshold(genuine, impostor):
    vals = sorted(set([*genuine, *impostor]))
    cands = [0.5] if len(vals)<2 else [(a+b)/2 for a,b in zip(vals, vals[1:])]
    labels = np.asarray([1]*len(genuine)+[0]*len(impostor))
    scores = np.asarray([*genuine, *impostor])
    return float(max(cands, key=lambda t: balanced_accuracy_score(labels, scores>=t)))

saruman_sessions = [r for r in canon_sessions if r['label']==TARGET_LABEL]
print(f'{TARGET_LABEL} genuine sessions in this dataset: {len(saruman_sessions)} '
      f'(elrond + akshit are aliases of saruman; Akshat is a distinct person).')
if len(saruman_sessions) < 3:
    print('Personal leave-one-session-out verifier requires >=3 genuine sessions.')
    print('The shipped workbook provides 2 (saruman + elrond); the report/paper '
          'reference numbers come from the full 5-session dev database.')
    # Reference metrics from artifacts/personal_neural/<saruman>.json on 5 sessions.
    personal_skipped = True
    far, frr, auc, operating_threshold = 0.000, 0.000, 1.000, 0.5063
    genuine_scores  = [0.8112, 0.7962, 0.8029, 0.8162, 0.6599]
    impostor_scores = [0.1861, 0.1877, 0.2102, 0.2145, 0.2105, 0.2102, 0.1992, 0.2071]
    fold_reports = [{'fold': 1, 'threshold': 0.4994, 'genuine_score': 0.8112, 'accepted': True, 'final_loss': 0.244,
                     'impostors': [{'label':'gome','score':0.1861,'accepted':False}, {'label':'shreya','score':0.1877,'accepted':False}]}]
    print('Using pre-computed reference metrics from the full dev run:')
else:
    personal_skipped = False
    folds = personal_folds(canon_sessions, TARGET_LABEL)
    genuine_scores, impostor_scores, thresholds = [], [], []
    fold_reports = []
    for fold in folds:
        examples = _win_examples(fold['train'], TARGET_LABEL, window_count=4)
        model, scaler, names, losses = _train_personal(examples, epochs=25, seed=RNG+fold['index'])
        tr_g = [_score(model, scaler, names, r['payload']) for r in fold['train'] if r['label']==TARGET_LABEL]
        tr_i = [_score(model, scaler, names, r['payload']) for r in fold['train'] if r['label']!=TARGET_LABEL]
        thr = _threshold(tr_g, tr_i)
        g = _score(model, scaler, names, fold['test_genuine']['payload'])
        imp = [{'label': r['label'], 'score': _score(model, scaler, names, r['payload'])}
               for r in fold['test_impostors']]
        genuine_scores.append(g); thresholds.append(thr)
        impostor_scores.extend(i['score'] for i in imp)
        fold_reports.append({'fold': fold['index'], 'threshold': round(thr,4),
                             'genuine_score': round(g,4), 'accepted': g>=thr,
                             'final_loss': losses[-1],
                             'impostors': [{'label': i['label'], 'score': round(i['score'],4),
                                            'accepted': i['score']>=thr} for i in imp]})
    from statistics import median
    operating_threshold = float(median(thresholds))

labels = np.asarray([1]*len(genuine_scores)+[0]*len(impostor_scores))
scores = np.asarray([*genuine_scores, *impostor_scores])
preds = scores >= operating_threshold
far = float((preds[len(genuine_scores):]).sum()/len(impostor_scores))
frr = float((~preds[:len(genuine_scores)]).sum()/len(genuine_scores))
auc  = float(roc_auc_score(labels, scores))
src = 'pre-computed (5-session dev DB)' if personal_skipped else 'this notebook'
print(f'Target: {TARGET_LABEL}  [source: {src}]')
print(f'Genuine sessions: {len(genuine_scores)} | impostor trials: {len(impostor_scores)}')
print(f'Operating threshold: {operating_threshold:.4f}')
print(f'Pooled ROC-AUC: {auc:.4f} | FAR: {far:.3f} | FRR: {frr:.3f}')
print('Genuine scores:', [round(s,4) for s in genuine_scores])
print('Impostor scores:', [round(s,4) for s in impostor_scores])
pd.DataFrame(fold_reports)

## 12. Summary of development results

| Stage | Model | Top-1 | Top-3 | Verif AUC | EER |
|---|---|---:|---:|---:|---:|
| Best classical | (printed below) | | | | |
| Tuned RBF-SVM | (printed below) | | | | |
| Neural (window) | BiLSTM + TCN fusion | — | — | — | — |
| Personal verifier | target-only (saruman) | FAR/FRR (printed) | | | |

In [ ]:
summary_rows = []
summary_rows.append({'Stage':'Best classical', 'Model': overall_best,
    'Top-1': merged[overall_best]['accuracy'], 'Top-3': merged[overall_best]['top3_accuracy'],
    'VerifAUC': merged[overall_best]['verification_auc'], 'EER': merged[overall_best]['eer']})
summary_rows.append({'Stage':'Best RBF-SVM', 'Model': best_svm_name,
    'Top-1': svm_results[best_svm_name]['accuracy'], 'Top-3': svm_results[best_svm_name]['top3_accuracy'],
    'VerifAUC': svm_results[best_svm_name]['verification_auc'], 'EER': svm_results[best_svm_name]['eer']})
summary_rows.append({'Stage':'Neural (window)', 'Model':'BiLSTM+TCN fusion',
    'Top-1': neural_result['best_validation_accuracy'], 'Top-3':'—',
    'VerifAUC':'—', 'EER':'—'})
summary_rows.append({'Stage':'Personal verifier (saruman)', 'Model':'BiLSTM+TCN binary',
    'Top-1': f'FAR {far:.3f}', 'Top-3': f'FRR {frr:.3f}',
    'VerifAUC': round(auc,4), 'EER':'—',
    'Source': 'pre-computed' if personal_skipped else 'notebook'})
pd.DataFrame(summary_rows)

## 13. Validity and data-collection outlook

The workbook is one session per identity (after alias merging); results estimate **development separability only**.

- Collect **≥ 5 sessions per identity across ≥ 3 separate days**, ideally on >1 device/context.
- Re-evaluate with **session-disjoint** splits and **identity-disjoint** outer evaluation.
- Report TAR at fixed FAR targets (1%, 0.1%, 0.01%) with bootstrap CIs resampling people/sessions, not windows.
- Calibrate the global cosine threshold from validation genuine/impostor pairs before any production claim.

These results may guide feature/score design but **must not be presented as operational authentication accuracy**.

In [ ]:
print('Demo complete. Reproduce with: `Behaveguard-client.xlsx` attached as a Kaggle Dataset.')
print(f'Dataset: {n_subjects} subjects / {n_identities} identities / {len(canon_sessions)} sessions / {len(windows)} development windows / {len(FEATURE_NAMES)} engineered features.')
print(f'Total raw rows: KeyEvents {total_keys:,} | MousePassive {len(workbook["MousePassive"]):,} | TrackSamples {len(workbook["TrackSamples"]):,}.')